In [1]:
import pandas as pd
import numpy as np
import os

I've tried to design this notebook so that we can 'run all' once the following cell, which gives a list containing all the 'Transee' data.

In [2]:
#Adding functions to read and write the correct dtypes for the csv files 
# Source - https://stackoverflow.com/a/50051542
# Posted by Aaron Brock, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-26, License - CC BY-SA 3.0

def to_csv(df, path):
    # Prepend dtypes to the top of df (from https://stackoverflow.com/a/43408736/7607701)
    df.loc[-1] = df.dtypes
    df.index = df.index + 1
    df.sort_index(inplace=True)
    # Then save it to a csv
    df.to_csv(path, index=False)

def read_csv(path):
    # Read types first line of csv
    dtypes = {key:value for (key,value) in pd.read_csv(path,    
              nrows=1).iloc[0].to_dict().items() if 'date' not in value}

    parse_dates = [key for (key,value) in pd.read_csv(path, 
                   nrows=1).iloc[0].to_dict().items() if 'date' in value]
    # Read the rest of the lines with the types from above
    return pd.read_csv(path, dtype=dtypes, parse_dates=parse_dates, skiprows=[1])

In [3]:
# Dictionary to store the desired dtypes for each column
# The datetime objects will have to be dealt with seperately, so we add those to the list of columns below

# First I will list all of the columns that appear in the data/raw_data/schedule_data with their desired dtypes
# Commenting out the ones that are going to be dropped so that we can just avoid reading them in
schedule_dtypes ={
    #"Unnamed: 0": "int64",
    #"Time": "string",
    "Vehicle": "float64",
    "Gap": "object", # I would really like for this to be a string but this caused some issues later so I will try to read it in as an object instead or just not specify the dtype? 
    "Headway": "object",  
    "Schedule": "object", 
    "Destination": "object",
    #"day": "datetime64[us]",  # deal with date type seperately            
    "day of the week": "int64",
    "stop": "string",
    #"Riders after stop": "float64"
}

# List all columns from dictionary keys plus any datetime columns
schedule_cols = ["day"] + list(schedule_dtypes.keys())

In [4]:
# Next read the data from each csv with the desired columns
schedule_data_dir = '../raw_data/schedule_data/'
year_dir = [x for x in os.listdir(schedule_data_dir) if x[:3] == '202']
print(year_dir)

['2025', '2026']


In [5]:
dataframes = []
for year in year_dir:
    data_files = os.listdir(os.path.join(schedule_data_dir, year))
    if 'with_stops' in data_files:
        csv_files = [file for file in os.listdir(os.path.join(schedule_data_dir,year,'with_stops')) if file[-4:] == '.csv']
        dataframes += [pd.read_csv(os.path.join(schedule_data_dir,year,'with_stops',file), usecols=schedule_cols) for file in csv_files]
    else:
        csv_files = [file for file in os.listdir(os.path.join(schedule_data_dir,year)) if file[-4:] == '.csv']
        dataframes += [pd.read_csv(os.path.join(schedule_data_dir,year,file), usecols=schedule_cols) for file in csv_files]

In [6]:
def find_datetime(day: str | float, time: str | float) -> pd.Timestamp | float:
    if type(day) != str or type(time) != str:
        return np.nan
    if time[0] == ' ':
        time = time[1:]
    return pd.Timestamp(f'{day} {time}')


In [7]:
for df in dataframes:
    df['Time'] = df.Schedule.str[-10:]
    df['scheduled time'] = df.apply(lambda x: find_datetime(day=x['day'], time=x['Time']), axis=1)
    df.drop(columns=['day', 'Time', 'day of the week'], inplace=True)

We're going to eventually split the data into eastbound vs westbound, so we'll add a column to address this.

In [8]:
def east_or_west(dest: str| float) -> str| float:
    if type(dest) != str:
        return np.nan
    dest = dest.lower()
    if 'east' in dest:
        return 'E'
    else:
        return 'W'


In [9]:
for df in dataframes:
    df['EB/WB'] = df['Destination'].apply(east_or_west)
    df.drop('Destination', axis=1, inplace=True)

In [10]:
def min_delay(schedule: str|float) -> int|float:
    if type(schedule) != str:
        return np.nan

    schedule = schedule.lower()
    min_marker = schedule.find(':')
    hour_marker = schedule[:min_marker].rfind(' ')
    if hour_marker == -1:
        hour_marker = 0

    minute = (int(schedule[hour_marker:min_marker])
              +int(schedule[min_marker+1:min_marker+3])/60)

    if 'ahead' in schedule:
        return -minute
    elif 'behind' in schedule:
        return minute
    else:
        return 0

for df in dataframes:
    df['min delay'] = df['Schedule'].apply(min_delay)
    df.drop('Schedule', axis=1, inplace=True)

Finally, 'cleaned_df' will be all of the dataframes merged together.

In [11]:
cleaned_df = pd.concat(dataframes, ignore_index=True)

In [12]:
cleaned_df.head()

,Vehicle,Gap,Headway,stop,scheduled time,EB/WB,min delay
0,4448.0,7:53,10:00,Grace St,2025-01-07 20:38:54,E,-6.016667
1,4605.0,13:37,10:00,Grace St,2025-01-07 20:48:54,E,-1.866667
2,4463.0,6:35,10:00,Grace St,2025-01-07 20:58:45,E,-4.733333
3,4415.0,12:05,10:00,Grace St,2025-01-07 21:08:54,E,-2.233333
4,4616.0,10:10,10:00,Grace St,2025-01-07 21:18:54,E,-2.066667


In [13]:
# We drop the instances where the min delay is over 2 hours, since that is most
# likely a cancellation rather than actual data.
cleaned_df = cleaned_df[cleaned_df['min delay'] < 120]

In [14]:
'12:00:00'.rfind(':')
'34:56'[-2:]

'56'

In [15]:
# Here we prune the data frame from segments where there is no gap info.
# First let's make sure that the Gap column is in time format.
# Turning the gap into a timestamp is problematic because it is sometimes negative.
# As well, sometimes the gap is sometimes a few hours (so it is of the format hh:mm:ss).
# Therefore, I elect to turn it into a float, where the units are in seconds.
def gap_to_seconds(gap: str|float) -> float:
    if type(gap) != str:
        return np.nan

    if gap.find('-') == -1:
        is_negative = False
    else:
        gap = gap[1:]
        is_negative = True

    if gap.count(':') > 1:
        hours = int(gap[:-6])
        minutes = int(gap[-5:-3])
        seconds = int(gap[-2:])
    else:
        hours = 0
        minutes = int(gap[:-3])
        seconds = int(gap[-2:])

    total_seconds = (hours*60 + minutes)*60 + seconds
    if is_negative:
        return -total_seconds
    else:
        return total_seconds



cleaned_df.Gap = cleaned_df.Gap.apply(gap_to_seconds)

n= 25
gap_na = cleaned_df.Gap.isna()
# For each index, this gives the largest size of a sequence of NaNs containing
# the index if the index is a NaN, and if the index is not a NaN, it gives the
# largest size of a sequence of non-NaNs.
s= gap_na.groupby(gap_na.diff().ne(0).cumsum()).transform('count')
# We only keep the data where the sequence of gaps of NaNs is smaller than n.
# If the Gap index is NaN, then the component is at most n.
cleaned_df = cleaned_df.loc[~(gap_na)|(s<=n)]

In [16]:
# I am going to elect to drop the times where there is no scheduling info
cleaned_df.dropna(axis=0, subset=['scheduled time'], inplace=True)

In [17]:
# Finally, we time order cleaned_df.
cleaned_df.sort_values(by=['scheduled time'], ignore_index=True, inplace=True)

In [18]:
cleaned_df.head()

,Vehicle,Gap,Headway,stop,scheduled time,EB/WB,min delay
0,4552.0,723.0,10:00,Woodfield Rd,2025-01-01 00:00:17,E,-5.350000
1,4400.0,684.0,11:15,Coxwell Ave at Gerrard St East,2025-01-01 00:00:39,W,-1.616667
2,4591.0,511.0,10:00,Euclid Ave,2025-01-01 00:00:40,E,-13.883333
3,4481.0,61.0,8:20,Bowmore Rd,2025-01-01 00:00:49,W,11.933333
4,4575.0,887.0,10:00,Bay St,2025-01-01 00:00:51,E,10.383333


The goal is to use the TTC summary data as the basis of our dataset, with columns appended
from cleaned_df. First we need to add a 'EB/WB' column.

In [19]:
summary_df = pd.read_csv('../data/TTC-Streetcar-Info/Streetcar_data_cleaned_up.csv')
remove_columns = ['Unnamed: 0', 'First dep NB or WB', 'First dep SB or EB', 'Last dep NB or WB', 'Last dep SB or EB', 'Route']
for column in remove_columns:
    if column in summary_df.columns:
        summary_df.drop(column, axis=1, inplace=True)

In [20]:
EB_or_WB= ['E', 'W']
stops = list(cleaned_df.stop.unique())

main_df = dict()
for dir in EB_or_WB:
    for stop in stops:
        main_df[dir, stop] = summary_df.copy()
        main_df[dir, stop]['EB/WB'] = [dir for _ in range(main_df[dir,stop].shape[0])]
        main_df[dir, stop]['stop'] = [stop for _ in range(main_df[dir,stop].shape[0])]

main_df = pd.concat(list(main_df.values()), ignore_index=True)
main_df = main_df.sort_values(by=['time period start', 'time period end'], ignore_index=True)
main_df.head()

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,RT dist (km),Interruption,time period start,time period end,EB/WB,stop
0,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E,Woodfield Rd
1,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E,Coxwell Ave at Gerrard St East
2,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E,Euclid Ave
3,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E,Bowmore Rd
4,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E,Bay St


In [21]:
# Let's turn the interruption 'nans' into 0-1s.
def binary_interruption(interruption: str|float) -> int:
    if type(interruption) != str:
        return 0
    else:
        return 1

main_df['Interruption'] = main_df['Interruption'].apply(binary_interruption)

# Turns the time periods into pandas timestamps
main_df['time period start'] = main_df['time period start'].apply(lambda x: pd.Timestamp(x))
main_df['time period end'] = main_df['time period end'].apply(lambda x: pd.Timestamp(x))

In [22]:
# We should filter main_df by the dates coming from cleaned_df.
earliest_timestamp = cleaned_df['scheduled time'].min()
latest_timestamp = cleaned_df['scheduled time'].max()
main_df = main_df[(main_df['time period start'] >= earliest_timestamp) & (main_df['time period end'] <= latest_timestamp)]

In [23]:
def count_bunch(df: pd.DataFrame) -> int:
    return len(df.Gap[df.Gap <= 120])

def count_gap(df: pd.DataFrame) -> int:
    return len(df.Gap[df.Gap > 19*60])

In [ ]:
number_bunch = dict()
number_gap = dict()
delay_amount = dict()

stops = list(cleaned_df.stop.unique())


def gather_data(direction, stop) -> None:
    filtered_main = main_df[(main_df['EB/WB'] == direction)
                           &(main_df['stop'] == stop)]
    filtered_cleaned = cleaned_df[(cleaned_df['EB/WB'] == direction)
                                & (cleaned_df['stop'] == stop)]
    for row in filtered_main.index:
        time_start = filtered_main['time period start'].loc[row]
        time_end = filtered_main['time period end'].loc[row]

        filtered_data = filtered_cleaned[
            filtered_cleaned['scheduled time'].between(time_start, time_end, inclusive='left')
             ]

        number_bunch[row]=count_bunch(filtered_data)
        number_gap[row]=count_gap(filtered_data)

        delay_amount[row]=sum(filtered_data['min delay'])

for direction in ['E','W']:
    for stop in stops:
        gather_data(direction, stop)

In [ ]:
list_bunch = [number_bunch[key] for key in sorted(number_bunch)]
list_gap = [number_gap[key] for key in sorted(number_gap)]
list_delay = [delay_amount[key] for key in sorted(delay_amount)]

main_df['bunch'] = list_bunch
main_df['gap'] = list_gap
main_df['total delay'] = list_delay

In [ ]:
# Finally, drop the 'Gap' column.
if 'Gap' in main_df.columns:
    main_df.drop('Gap', axis=1, inplace=True)

In [28]:
# Now we make sure that date is a timestamp.
main_df['date'] = main_df['date'].apply(lambda x: pd.Timestamp(x))
main_df.head()

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,RT dist (km),Interruption,time period start,time period end,EB/WB,stop,bunch,gap,total delay
524140,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,E,Woodfield Rd,1,0,-40.283333
524141,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,E,Coxwell Ave at Gerrard St East,0,0,0.000000
524142,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,E,Euclid Ave,0,0,-28.800000
524143,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,E,Bowmore Rd,1,0,-41.933333
524144,2025-01-01,16,10:00,150,10,12.1,0,0,30.13,0,2025-01-01 04:00:00,2025-01-01 09:00:00,E,Bay St,0,0,-44.816667


Since our data is big, we split the dataset into time of day/ weekday vs weekend.

In [29]:
# name of the time period column (it's very long)
TIME = [x for x in summary_df.columns if 'Time' in x][0]
# name of the week/sat/sun column (it's also very long)
WEEK = [x for x in summary_df.columns if 'Week' in x][0]

week_translator = {0: 'weekday', 1: 'saturday', 2: 'sunday'}

df_split = dict()
for time in range(5):
    for week in range(3):
        df_split[time, week] = main_df[(main_df[TIME] == time) & (main_df[WEEK] == week)]

Finally, let's write all the data into their own csv files. The final files are not that large because
the rows are split by date/time period.

In [30]:
for key, df in df_split.items():
    to_csv(df,f'../data/schedule_data/processed_data/by_weekday/with_stops/schedule_data_{week_translator[key[1]]}_{key[0]}.csv')

/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_78586/2523171216.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[-1] = df.dtypes
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_78586/2523171216.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.sort_index(inplace=True)
/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_78586/2523171216.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.

# Adding a 'by date' version of the data.
Here we create a separate data file that forms a 'by date' version of the data. Since the service periods are removed, we choose to take an average or take a sum over the day depending on the feature.

In [31]:
daily_df = dict()
stops = main_df['stop'].unique()

for direction in ['E', 'W']:
    for stop in stops:
        daily_df[direction, stop] = pd.DataFrame()

In [32]:
def average_feature(date, feature, df):
    length = len(df[df['date'] == date][feature])
    if length==0:
        return 0
    else:
        return sum(df[df['date'] == date][feature])/length

def count_feature(date, feature, df):
    if len(df[df['date'] == date][feature].tolist())==0:
        return 0
    else:
        return sum(df[df['date'] == date][feature])

# Separate function to deal with interruptions
def count_interruption(date, df):
    if len(df[df['date'] == date]['Interruption']) == 0:
        return 0
    else:
        return max(df[df['date'] == date]['Interruption'])


In [33]:
first_date = main_df['date'].min()
last_date = main_df['date'].max()

timeframe = (last_date - first_date).days

for dir in ['E','W']:
    for stop in stops:
        daily_df[dir, stop]['date'] = [first_date+pd.Timedelta(days=k) for k in range(timeframe)]
        daily_df[dir, stop]['EB/WB'] = [dir for _ in range(timeframe)]
        daily_df[dir, stop]['stop'] = [stop for _ in range(timeframe)]

In [34]:
# Make a choice of which columns are counted and which are averaged
count_columns = ['bunch', 'gap', 'total delay']
average_columns = ['No. of Veh', 'Run time (min)', 'Term time (min)', 'Avg. spd (km/h)', 'RT dist (km)']

In [35]:
for dir in ['E','W']:
    for stop in stops:
        current_df = main_df[(main_df['EB/WB'] == dir)&(main_df['stop'] == stop)]
        for column in count_columns:
            daily_df[dir, stop][column] = daily_df[dir, stop].apply(lambda x: count_feature(x['date'], column, current_df), axis=1)

        for column in average_columns:
            daily_df[dir, stop][column] = daily_df[dir, stop].apply(lambda x: average_feature(x['date'], column, current_df), axis=1)

        daily_df[dir, stop]['Interruption'] = daily_df[dir, stop].apply(lambda x: count_interruption(x['date'], current_df), axis=1)

daily_df = pd.concat(list(daily_df.values()), ignore_index=True)

In [36]:
daily_df = daily_df.sort_values(by='date', axis=0)
daily_df.head()

,date,EB/WB,stop,bunch,gap,total delay,No. of Veh,Run time (min),Term time (min),Avg. spd (km/h),RT dist (km),Interruption
0,2025-01-01,E,Woodfield Rd,12,7,-632.833333,16.4,153.4,10.6,11.92,30.13,0
23265,2025-01-01,E,Blackburn St,0,0,0.000000,16.4,153.4,10.6,11.92,30.13,0
49491,2025-01-01,W,High Park Loop,0,0,0.000000,16.4,153.4,10.6,11.92,30.13,0
3807,2025-01-01,E,Gerrard St East at Beaton Ave,14,10,-608.650000,16.4,153.4,10.6,11.92,30.13,0
19458,2025-01-01,E,Roncesvalles Ave,1,1,-198.516667,16.4,153.4,10.6,11.92,30.13,0


In [37]:
daily_df.to_csv(f'../data/schedule_data/processed_data/by_day/schedule_data_with_stops.csv', index=False)